In [10]:
import time
from sage.all import *
import copy

![Schéma du LFSR demandé]("img/exo1_a.png")

# 1 - Attaque générique par compromis temps-mémoire
## Exercice 6 - Création d'un LFSR

In [11]:
L = [[0, 1, 1, 0, 0, 0, 0, 1, 1], [0, 1, 1, 1, 0, 0, 0], [0, 0, 1, 1, 1, 0, 1, 1]]
realStream  = [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0]
k = GF(Integer(5)); type(k)
    
def LFSR_L1(state):
    L1 = state[0]
    new = (L1[0] ^^ L1[5])
    z1 = L1.pop(0)
    L1.append(new)
    state[0]=L1
    return z1,state

def LFSR_L2(state):
    L2 = state[1]
    new = (((L2[6] ^^ L2[4]) ^^ L2[2]) ^^ L2[0])
    z2 = L2.pop(0)
    L2.append(new)
    state[1] = L2
    return z2, state

def LFSR_L3(state):
    L3 = state[2]
    new = (((L3[0] ^^ L3[1]) ^^ L3[6]) ^^ L3[7])
    z3 = L3.pop(0)
    L3.append(new)
    state[2]=L3
    return z3,state
    
def f(z1,z2,z3):
    return (((z1 & z2) ^^ (z2 & z3)) ^^ (z3))

def LFSR_comb(stat,stream):
    z1, stat = LFSR_L1(stat)
    z2, stat = LFSR_L2(stat)
    z3, stat = LFSR_L3(stat)
    z = f(z1,z2,z3)
    new_stream = stream + [z]
    return stat,new_stream

stat = L
stream = []
key = []
for i in range(0,20):
    stat,stream = LFSR_comb(stat,stream)
    print("LFSR - Flot : ",stream)
    if realStream[i] != stream[i]:
        print("ERREUR DE FLOT - Vérifiez l'algorithme ")
        print("RLFS : ",realStream)

LFSR - Flot :  [0]
LFSR - Flot :  [0, 1]
LFSR - Flot :  [0, 1, 1]
LFSR - Flot :  [0, 1, 1, 0]
LFSR - Flot :  [0, 1, 1, 0, 1]
LFSR - Flot :  [0, 1, 1, 0, 1, 0]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1]
LFSR - Flot :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0]


## Exercice 6 BIS avec GF

In [12]:
L = [[0, 1, 1, 0, 0, 0, 0, 1, 1], [0, 1, 1, 1, 0, 0, 0], [0, 0, 1, 1, 1, 0, 1, 1]]
L = [vector(GF(2), reg).list() for reg in L]     # <- ligne ajoutee : passage en GF(2)

realStream  = [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0]

def LFSR_L1(state):
    L1 = state[0]
    new = (L1[0] + L1[5])                        # ^ -> +
    z1 = L1.pop(0)
    L1.append(new)
    state[0]=L1
    
    return z1,state

def LFSR_L1_fixed(state):
    new = (state[0] + state[5])                        # ^ -> +
    z1 = state.pop(0)
    state.append(new)
    return z1,state

def LFSR_L2(state):
    L2 = state[1]
    new = L2[6] + L2[4] + L2[2] + L2[0]          # ^ -> +
    z2 = L2.pop(0)
    L2.append(new)
    state[1] = L2
    
    return z2, state

def LFSR_L3(state):
    L3 = state[2]
    new = L3[0] + L3[1] + L3[6] + L3[7]          # ^ -> +
    z3 = L3.pop(0)
    L3.append(new)
    state[2]=L3
    return z3,state
    
def LFSR_L3_fixed(state):
    new = (state[0] + state[1] + state[6] + state [7])                        # ^ -> +
    z3 = state.pop(0)
    state.append(new)
    return z3,state

def f(z1,z2,z3):
    return (z1 * z2) + (z2 * z3) + (z3)          # & -> * , ^ -> +

def LFSR_comb(stat,stream):
    z1, stat = LFSR_L1(stat)
    z2, stat = LFSR_L2(stat)
    z3, stat = LFSR_L3(stat)
    z = f(z1,z2,z3)
    new_stream = stream + [z]
    return stat,new_stream

stat = copy.deepcopy(L)
stream = []
key = []
for i in range(0,100):
    stat,stream = LFSR_comb(stat,stream)
print("LFSR COMBINÉ - Flot produit : ",stream)

LFSR COMBINÉ - Flot produit :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1]


## Exercice 7 - Attaque exhaustive

In [13]:
'''false_state = true
L_TEST = []
while (false_state):
    for L1 in VectorSpace(GF(2),9):
        if not (false_state):
            break;
        for L2 in VectorSpace(GF(2),7):
            if not(false_state):
                break;
            for L3 in VectorSpace(GF(2),8):
                L_TEST = [L1.list(),L2.list(), L3.list()]
                fake_stat = L_TEST
                fake_stream = []
                key = []
                for i in range(0,100):
                    fake_stat,fake_stream = LFSR_comb(fake_stat,fake_stream)
                    if stream[i] != fake_stream[i]:
                        print("ERREUR DE FLOT - Vérifiez l'algorithme ")
                        print("RLFS : ",realStream)
                        false_state = true
                        break;     
                    else:
                        false_state = false
print("L TEST IS : ", L_TEST)
'''


''' Parallelized code  '''
@parallel(ncpus=15)
def search_for_L1(L1_vec):
    l1 = L1_vec.list()
    for L2 in VectorSpace(GF(2), 7):
        for L3 in VectorSpace(GF(2), 8):
            L_TEST = [list(l1), L2.list(), L3.list()]
            fake_stat = [list(l1), L2.list(), L3.list()]
            fake_stream = []
            match = True
            for i in range(len(stream)):
                fake_stat, fake_stream = LFSR_comb(fake_stat, fake_stream)
                if stream[i] != fake_stream[-1]:
                    match = False
                    break
            if match:
                return L_TEST
    return None

l1_candidates = list(VectorSpace(GF(2), 9))

t0 = time.time()
i_count = 0
trouve = None
for entry in search_for_L1(l1_candidates):
    (args, kwargs), res = entry
    i_count += 1
    elapsed = time.time() - t0
    print(f"\rATTAQUE EXHAUSTIVE - Itération n°{i_count}/{len(l1_candidates)} — {elapsed:.1f} s écoulées", end="", flush=True)
    if res is not None:
        trouve = res

print()
if trouve is not None:
    print("ATTAQUE EXHAUSTIVE - Etat trouve :", trouve)
else:
    print("Aucun etat trouve.")

ATTAQUE EXHAUSTIVE - Itération n°512/512 — 33.7 s écoulées
ATTAQUE EXHAUSTIVE - Etat trouve : [[0, 1, 1, 0, 0, 0, 0, 1, 1], [0, 1, 1, 1, 0, 0, 0], [0, 0, 1, 1, 1, 0, 1, 1]]


## Exercice 8 - Attaque temps/mémoire

In [14]:
l = 24
N_base = int(sqrt(2 ** l).numerical_approx())

def gen_state(N) :
    dic = {}
    for i in range (N):
        L1_gen_state = list(VectorSpace(GF(2),9).random_element())
        L2_gen_state = list(VectorSpace(GF(2),7).random_element())
        L3_gen_state = list(VectorSpace(GF(2),8).random_element())
        L_3 = [L1_gen_state,L2_gen_state,L3_gen_state]
        L_3_base = copy.deepcopy([L1_gen_state,L2_gen_state,L3_gen_state])
        stream = []
        state = []
        for j in range(0,50):
            state,stream = LFSR_comb(L_3,stream)
        dic[tuple(stream)] = L_3_base
    return dic

def gen_state_basic(N,init_state):
    stat = copy.deepcopy(init_state)
    stream = []
    key = []
    for i in range(0,N+50):
        stat,stream = LFSR_comb(stat,stream)
    return stream
    
def time_memory_attack_k(N,dic) :
    t0 = time.time()
    i_count = 0
    real_init_state = copy.deepcopy(L)
    stream = gen_state_basic(N,real_init_state)
    fetching_state = true
    for j in range(N-49):
        tuple_stream = tuple(stream[j:j+50])
        if tuple_stream in dic:
            init_stream = tuple_stream
            return {
                "init_state" : dic[init_stream],
                "linked_stream" : init_stream,
                "interval" : j,
            }
        elapsed = time.time() - t0
        #print(f"\rItération n°{i_count}/ — {elapsed:.1f} s écoulées", end="", flush=True)
        i_count += 1
    print("No state found")                 

gen_dic = gen_state(N_base)
result_k = time_memory_attack_k(N_base,gen_dic)
if result_k:
    print("TIME/MEMORY ATTACK - Un état probable a été retrouvé, voici ses infos")
    print("TIME/MEMORY ATTACK - init state:", result_k["init_state"])
    print("TIME/MEMORY ATTACK - offset:", result_k["interval"])

TIME/MEMORY ATTACK - Un état probable a été retrouvé, voici ses infos
TIME/MEMORY ATTACK - init state: [[1, 0, 1, 0, 0, 1, 0, 0, 1], [1, 0, 1, 0, 0, 0, 0], [1, 1, 0, 1, 1, 0, 1, 0]]
TIME/MEMORY ATTACK - offset: 965


In [15]:
real_init_state = copy.deepcopy(L)
real_stream = gen_state_basic(N_base, real_init_state)

In [16]:
if result_k:
    guessed_init_state = result_k["init_state"]
    interval = result_k["interval"]
    real_stream = gen_state_basic(N_base, real_init_state)
    fake_stream = gen_state_basic(N_base-interval, guessed_init_state)
    truncated_stream = real_stream[interval:]
    
    
    if fake_stream != truncated_stream:
        print("Stream are not the same")
    else:
        print("TIME/MEMORY ATTACK -  État retrouvé -> Bien joué!!!")
else:
    print("TIME/MEMORY ATTACK - Aucun État initial à été retrouvé!")

TIME/MEMORY ATTACK -  État retrouvé -> Bien joué!!!


# 2 - Attaque par correlation

## Exercice 9

In [17]:
# Code question A
real_init_state = copy.deepcopy(L)
newFlow = gen_state_basic(50,real_init_state)

def ex_LFSR_1() :
    gen_stream = []
    all_gen_streams={}
    allStates = VectorSpace(GF(2),9).list()
    for i in allStates:
        current_stat = i.list()
        for j in range(100):
            z_1, current_stat = LFSR_L1_fixed(current_stat)
            gen_stream.append(z_1)
        all_gen_streams[tuple(i)] = gen_stream
        gen_stream = []
    return all_gen_streams

def correlationAttack(generatedStreams,knownFlow):
    possible_correlated_states = []
    streamCount = 0
    best_correlation = 0
    best_correlated_state = []
    for k in generatedStreams:
        selected_stream = generatedStreams[k]
        i = 0
        equivalent_bit_count = 0
        for bit in selected_stream:
            equivalent_bit_count += 1 if (bit == knownFlow[i]) else 0
            i+= 1
        percentage = (equivalent_bit_count / len(selected_stream)).n()
        if percentage >= best_correlation:
            best_correlated_state = k
            best_correlation = percentage
        streamCount += 1
    
    return best_correlated_state,best_correlation

l1_gen_streams = ex_LFSR_1()

bestL1_state, bestL1_corr = correlationAttack(l1_gen_streams,newFlow)

print("CORRELATION ATTACK - Meilleur État initial L1 est ", bestL1_state, "avec" ,bestL1_corr, "% de correlation")

CORRELATION ATTACK - Meilleur État initial L1 est  (0, 1, 1, 0, 0, 0, 0, 1, 1) avec 0.710000000000000 % de correlation


### Question B
En prenant $$P(z_2=z)$$ on observe que cette probabilité est de 1/2, indiquant qu'il n'y a aucune correlation, il n'est donc pas utile d'effectuer une attaque par correlation sur ce registre

### Question C
En prenant $$(P(z_3=z)$$, on peut voir que la probabilité est de 3/4, indiquant qu'il y'a une forte correlation entre les deux, donc nous pouvons implémenter une attaque par correlation

In [18]:
def ex_LFSR_3() :
    gen_stream = []
    all_streams={}
    allStates = VectorSpace(GF(2),8).list()
    for i in allStates:
        current_stat = i.list()
        for j in range(100):
            z_3, current_stat = LFSR_L3_fixed(current_stat)
            gen_stream.append(z_3)
        all_streams[tuple(i)] = gen_stream
        gen_stream = []
    return all_streams
    
bestL3_state, bestL3_corr = correlationAttack(ex_LFSR_3(),newFlow)

print("CORRELATION ATTACK - Le meilleur état initial L3 est ", bestL3_state, "avec" ,bestL3_corr, "% de corrélation")

CORRELATION ATTACK - Le meilleur état initial L3 est  (0, 0, 1, 1, 1, 0, 1, 1) avec 0.710000000000000 % de corrélation


### Question D
La seule information restante est le registre L2, il suffit donc d'itérer sur celui-ci de manière exhaustive (512 combinaisons)

In [19]:
L2_gen_states = VectorSpace(GF(2),7)

def find_key():
    for init_L2_state in L2_gen_states:
        guessed_init_state = [list(bestL1_state), init_L2_state.list(), list(bestL3_state)]
        gis = copy.deepcopy(guessed_init_state)
        generated_flow = gen_state_basic(50, gis)
        if generated_flow == newFlow:
            return guessed_init_state

realInitState = find_key()
print("CORRELATION ATTACK - l'état intial à été retrouvé : ",realInitState)

CORRELATION ATTACK - l'état intial à été retrouvé :  [[0, 1, 1, 0, 0, 0, 0, 1, 1], [0, 1, 1, 1, 0, 0, 0], [0, 0, 1, 1, 1, 0, 1, 1]]


### Question E 
En ce qui concerne la complexité en temps, au lieu d'effectuer $$O(2^{l_1}\times 2^{l_2}\times 2^{l_3})=O(2^{l_1+l_2+l_3})$$ ce qui donne près de 16 million d'itérations sur ce cas, ici on passe à une complexité en temps de $$O(2^{l_1}+2^{l_2}+2^{l_3})$$. Donnant ainsi 896 itérations sur ce cas là seulement
Cette attaque n'as besoin également que d'un espace mémoire O(1)
Enfin, il est nécessaire d'avoir plus de 100 bit de flot connu pour être en mesure de correctement établir les corrélations entre les flots de chaque LFSR unitaire

# 3 - Attaque Algébrique
## Exercice 11

In [20]:
def base_LFSR(state):
    new = (state[0] + state[4] + state[6] + state[7] + state[10] + state[18] + state[21] + state[22])
    state.pop(0)
    state.append(new)
    return state
    
def filter(state):
    filtered_state = state[0] + state[10] + (state[2]*state[13]) + (state[3]*state[19]) + (state[11]*state[22])
    return filtered_state
    
def LFSR_filt(stat,stream):
    z = filter(stat)
    stat = base_LFSR(stat)
    new_stream = stream + [z]
    return stat,new_stream

filt_init_state = [1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1]
filt_init_state = vector(GF(2), filt_init_state).list()
  = [1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0]
stat = copy.deepcopy(filt_init_state)
generated_stream = []
for i in range(30):
    stat, generated_stream = LFSR_filt(stat, generated_stream) 
if generated_stream == real_stream:
    print("Exact stream !")
else:
    print("Wrong stream")
    print("Length", len(generated_stream), "vs" ,len(real_stream))
    print(generated_stream)
    print(real_stream)

Exact stream !


## Exercice 12

In [24]:
'''trouve = None
for candidate_vec in VectorSpace(GF(2), 23):
    candidate = candidate_vec.list()
    fake_stat = copy.deepcopy(candidate)
    fake_stream = []
    match = True
    for i in range(len(real_stream)):
        fake_stat, fake_stream = LFSR_filt(fake_stat, fake_stream)
        if real_stream[i] != fake_stream[-1]:
            match = False
            break
    if match:
        trouve = candidate
        break

print("Etat trouve : ", trouve)'''


@parallel(ncpus=15)
def search_prefix(prefix_vec):
    prefix = prefix_vec.list()
    for suffix_vec in VectorSpace(GF(2), 14):
        candidate = prefix + suffix_vec.list()
        fake_stat = copy.deepcopy(candidate)
        fake_stream = []
        match = True
        for i in range(len(real_stream)):
            fake_stat, fake_stream = LFSR_filt(fake_stat, fake_stream)
            if real_stream[i] != fake_stream[-1]:
                match = False
                break
        if match:
            return candidate
    return None

prefixes = list(VectorSpace(GF(2), 9))
t0 = time.time()
i_count = 0
trouve = None
for entry in search_prefix(prefixes):
    (args, kwargs), res = entry
    i_count += 1
    elapsed = time.time() - t0
    print(f"\rATTAQUE EXHAUSTIVE - Itération n°{i_count} — {elapsed:.1f} s écoulées", end="", flush=True)
    if res is not None:
        trouve = res

if trouve == filt_init_state:
    print("\nEtat trouve : ", trouve)

ATTAQUE EXHAUSTIVE - Itération n°512 — 12.5 s écoulées
Etat trouve :  [1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1]


## Exercice 13

- Sachant que la formule donnant le nombre de Monome maximum est la suivante $$\sum_{d=1}^{deg(f)} \binom{l}{d}$$
Avec ici $$deg(f)=2$$ et $$l=23$$ on retrouve la formule suivante $$\frac{23!}{\left( 22 \right) !} +\frac{23!}{\left( 21! \right) \cdot 2}$$
Il y'a alors 276 Monômes possibles

### Question B

In [22]:
N_MONOMS = 276
var = list(BooleanPolynomialRing(23,"s").gens())
print(var)
equation = var[0] * (var[1] + var[22] * (var[0] + var[11]))
print(equation)


[s0, s1, s2, s3, s4, s5, s6, s7, s8, s9, s10, s11, s12, s13, s14, s15, s16, s17, s18, s19, s20, s21, s22]
s0*s1 + s0*s11*s22 + s0*s22


### Question C

In [33]:
# Real stream that was generated from the real key
stat = copy.deepcopy(filt_init_state)
original_stream = []
for i in range(N_MONOMS):
    stat, original_stream = LFSR_filt(stat, original_stream) 
print("algebraic attack - Original stream", original_stream)

# Fake stream which contains only values
stat = copy.deepcopy(var)
var_stream = []
for i in range(N_MONOMS):
    stat, var_stream = LFSR_filt(stat, var_stream) 
print("algebraic attack - Fake stream", var_stream[:5])
print("algebraic attack - Fake stream length", len(original_stream))

algebraic attack - Original stream [1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0]
algebraic attack - Fake stream [s0 + s2*s13 + s3*s19 + s10 + s11*s22, s0*s12 + s1 + s3*s14 + s4*s12 + s4*s20 + s6*s12 + s7*s12 + s10*s12

### Question D

In [86]:
variable_dic = {}
for i in var_stream:
    for j in i:
        if j.degree() == 1:
            variable_dic[j.index()] = j
        elif j not in variable_dic.values():
            max_index = max(variable_dic.keys())
            new_index = 23 if (max_index <= 22) else max_index+1
            variable_dic[new_index] = j
print("Algebraic attack - Variable dic", variable_dic)

Algebraic attack - Variable dic {0: s0, 23: s2*s13, 24: s3*s19, 10: s10, 25: s11*s22, 26: s0*s12, 1: s1, 27: s3*s14, 28: s4*s12, 29: s4*s20, 30: s6*s12, 31: s7*s12, 32: s10*s12, 11: s11, 33: s12*s18, 34: s12*s21, 35: s12*s22, 36: s0*s13, 37: s1*s13, 2: s2, 38: s4*s13, 39: s4*s15, 40: s5*s13, 41: s5*s21, 42: s6*s13, 43: s8*s13, 44: s10*s13, 45: s11*s13, 12: s12, 46: s13*s18, 47: s13*s19, 48: s13*s21, 49: s1*s14, 50: s2*s14, 3: s3, 51: s5*s14, 52: s5*s16, 53: s6*s14, 54: s6*s22, 55: s7*s14, 56: s9*s14, 57: s11*s14, 58: s12*s14, 13: s13, 59: s14*s19, 60: s14*s20, 61: s14*s22, 62: s0*s7, 63: s0*s15, 64: s2*s15, 65: s3*s15, 66: s4*s7, 4: s4, 67: s6*s7, 68: s6*s17, 69: s7*s10, 70: s7*s18, 71: s7*s21, 72: s7*s22, 7: s7, 73: s8*s15, 74: s12*s15, 75: s13*s15, 14: s14, 76: s15*s18, 77: s15*s20, 78: s15*s22, 79: s0*s8, 80: s0*s16, 81: s1*s8, 82: s1*s16, 83: s3*s16, 84: s4*s8, 85: s5*s8, 5: s5, 86: s6*s8, 87: s6*s16, 88: s7*s16, 89: s8*s10, 90: s8*s11, 91: s8*s18, 92: s8*s19, 93: s8*s21, 8: s8, 94

### Question E

In [90]:
M = MatrixSpace(GF(2),N_MONOMS)()


index_of = {mono: idx for idx, mono in variable_dic.items()}
for i, equation in enumerate(var_stream):
    for term in equation:
        if term.degree() == 0:
            continue          # le terme constant n'est pas une colonne de M
        j = index_of[term]
        M[i, j] = 1

### Question F

In [94]:
v = VectorSpace(GF(2),N_MONOMS)()
for i in range(N_MONOMS):
    v[i] = original_stream[i]

(1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0)


### Question G

In [98]:
x = M.solve_right(v)
print(x)
print("Etat initial trouvé : ", x[:23])
print("Véritable état initial : ", filt_init_state)

(1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0)
Etat initial trouvé :  (1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1)
Véritable état initial :  [1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 

### Question H

L'attaque exhaustive présente une complexité en temps exponentielle en $O(2^n)$, alors que l'attaque algébrique la ramène à une complexité polynomiale en $O(n^6)$, au prix d'une consommation mémoire plus élevée ($O(n^4)$ contre $O(1)$) et d'une quantité de flot connu nécessaire plus importante ($O(n^2)$ contre $O(n)$). Pour $n=23$, les deux approches restent praticables en quelques secondes, l'avantage de l'attaque algébrique n'étant donc pas perceptible à cette échelle. Il devient en revanche déterminant pour des registres de taille réaliste (64 ou 128 bits), où l'exhaustive devient infaisable alors que l'algébrique demeure calculable en un temps raisonnable, illustrant ainsi l'intérêt cryptanalytique de passer d'une complexité exponentielle à une complexité polynomiale.